# EDA Goodreads - History & Biography

Este notebook aplica el mismo análisis exploratorio a History & Biography para comparar calidad, ratings, temporalidad, duplicados y outliers con la categoría Fantasy & Paranormal.

In [ ]:
from pathlib import Path
import os, sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.config import CATEGORIES
from src.utils.io import read_jsonl_sample
from src.utils.cleaning import clean_books, clean_interactions
from src.utils.eda import missing_summary, duplicate_summary, iqr_outlier_summary

cfg = CATEGORIES['history_biography']
sns.set_theme(style='whitegrid')

## Carga de muestra

La exploración usa una muestra suficientemente grande para perfilar columnas y distribuciones sin cargar todas las interacciones en memoria.

In [ ]:
books_raw = read_jsonl_sample(cfg.books_file, nrows=50_000)
interactions_raw = read_jsonl_sample(cfg.interactions_file, nrows=250_000)
books = clean_books(books_raw)
interactions = clean_interactions(interactions_raw)
books.shape, interactions.shape

## Esquema, nulos y strings vacíos

Los campos de Goodreads mezclan strings numéricos, listas anidadas y strings vacíos. El pipeline limpia esos casos antes de escribir Parquet.

In [ ]:
display(missing_summary(books_raw).head(20))
display(missing_summary(interactions_raw).head(20))

## Ratings

Se separa el rating explícito de usuario (`rating_clean`) de la ausencia de rating (`rating = 0`). Esto evita entrenar modelos interpretando ceros como malas calificaciones.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(books['average_rating'].dropna(), bins=30, ax=axes[0])
axes[0].set_title('Average rating por libro')
sns.countplot(data=interactions, x='rating', ax=axes[1])
axes[1].set_title('Rating de usuario, incluyendo 0 = sin rating')
plt.tight_layout()
interactions[['rating', 'rating_clean', 'rating_missing']].head()

## Publicaciones en el tiempo

El análisis temporal ayuda a detectar años imposibles, concentración editorial y diferencias entre obras históricas antiguas y ediciones modernas.

In [ ]:
year_counts = books['publication_year'].dropna().astype(int).value_counts().sort_index()
year_counts.loc[year_counts.index.between(1800, 2025)].plot(figsize=(12, 4), title='Libros por año de publicación')
plt.xlabel('Año')
plt.ylabel('Cantidad')

## Correlaciones y outliers

Los outliers se reportan mediante IQR y percentiles. No se eliminan por defecto; para modelado se crean columnas capadas al p99 y se preservan las variables originales.

In [ ]:
num_cols = ['average_rating', 'ratings_count', 'text_reviews_count', 'num_pages', 'publication_year']
display(books[num_cols].corr(numeric_only=True))
sns.heatmap(books[num_cols].corr(numeric_only=True), annot=True, cmap='vlag', center=0)
plt.title('Correlaciones de metadatos')
plt.show()
display(iqr_outlier_summary(books, num_cols))

## Duplicados

Se revisan duplicados en llaves naturales antes de curar. La lógica final está centralizada en `src.utils.cleaning`.

In [ ]:
display(duplicate_summary(books_raw, interactions_raw))